# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Authors: {[a['@id'] for a in metadata.author]}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** In Croissant datasets, each entity (record set, field, column) is referenced by its `@id`. Here we list available record sets in this dataset, then inspect records.

In [ ]:
# Retrieve record set IDs from metadata
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print('No record sets found in metadata. Attempting to query record sets through dataset.records()...')
    # Attempt: get available record_set IDs from the schema if they are not in metadata
    try:
        from urllib.request import urlopen
        schema = json.load(urlopen(croissant_url))
        # Seek 'recordSet' top-level or as '@graph' objects
        def find_record_sets(obj):
            recs = []
            if isinstance(obj, dict):
                if obj.get('@type') in ['RecordSet', 'cr:RecordSet']:
                    recs.append(obj['@id'])
                for v in obj.values():
                    recs.extend(find_record_sets(v))
            elif isinstance(obj, list):
                for item in obj:
                    recs.extend(find_record_sets(item))
            return recs
        record_sets = find_record_sets(schema)
        print(f"Record set IDs detected in schema: {record_sets}")
    except Exception as e:
        print('Could not retrieve record set IDs:', e)
else:
    record_sets_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]
    print(f"Record sets listed in metadata: {record_sets_ids}")

# Show records for each discovered record set
for record_set_id in record_sets:
    if isinstance(record_set_id, dict) and '@id' in record_set_id:
        record_set_id = record_set_id['@id']
    print(f"Sample records for record set {record_set_id}:")
    try:
        for x in dataset.records(record_set=record_set_id):
            print(x)
            break  # Print only one record per record set for overview
    except Exception as e:
        print(f"Could not load records for {record_set_id}:", e)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above. If more than one record set is available, load all.

**Note:** All identifiers are referenced with their `@id` property.

In [ ]:
# Store record set IDs
# If not already extracted above, try to find them here again
if not record_sets:
    # Try from schema
    from urllib.request import urlopen
    schema = json.load(urlopen(croissant_url))
    def find_record_sets(obj):
        recs = []
        if isinstance(obj, dict):
            if obj.get('@type') in ['RecordSet', 'cr:RecordSet']:
                recs.append(obj['@id'])
            for v in obj.values():
                recs.extend(find_record_sets(v))
        elif isinstance(obj, list):
            for item in obj:
                recs.extend(find_record_sets(item))
        return recs
    record_sets_ids = find_record_sets(schema)
else:
    record_sets_ids = [rs['@id'] if isinstance(rs, dict) and '@id' in rs else rs for rs in record_sets]

dataframes = {}
for record_set_id in record_sets_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id}. Columns: {df.columns.tolist()}")
        print(df.head())
    except Exception as e:
        print(f"Failed to load records for {record_set_id}: {e}")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**All fields referenced by `@id`.**

Let's select a record set and numeric field for analysis.

In [ ]:
# Choose a record set and a numeric field
# Inspect what columns are available

# List available DataFrames
for rs_id, df in dataframes.items():
    print(f"Record set: {rs_id}, Columns: {df.columns.tolist()}")

# Example: Let's assume 'Table2' contains hormone levels as numeric fields
# Replace below with exact @id of record set for Table2 (the hormone level table) and field @id

record_set_for_eda = None
numeric_field_id = None
# Find an example record set and numeric field
for rs_id, df in dataframes.items():
    num_cols = [col for col in df.columns if 'hormone' in col.lower() or df[col].dtype in ['float64', 'int64']]
    if num_cols:
        record_set_for_eda = rs_id
        numeric_field_id = num_cols[0]
        break

if record_set_for_eda and numeric_field_id:
    print(f"Example record set selected for EDA: {record_set_for_eda}, numeric field: {numeric_field_id}")
else:
    print("No suitable numeric field or record set found for EDA.")

# Proceed if found
if record_set_for_eda and numeric_field_id:
    df = dataframes[record_set_for_eda]
    # Filter records with numeric field > threshold
    threshold = 10
    if numeric_field_id in df.columns:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by another field if available
        group_field_id = None
        for col in df.columns:
            if 'patient' in col.lower() or 'phenotype' in col.lower():
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below, we plot a histogram of the selected numeric field and a boxplot by group (if available).

In [ ]:
if record_set_for_eda and numeric_field_id:
    df = dataframes[record_set_for_eda]
    # Histogram
    plt.figure(figsize=(6,4))
    df[numeric_field_id].dropna().hist(bins=10)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group if available
    if group_field_id:
        plt.figure(figsize=(6,4))
        df.boxplot(column=numeric_field_id, by=group_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle('')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

In this notebook, we demonstrated how to load and explore a FAIR^2 dataset using the `mlcroissant` library. Key steps included:

- Loading Croissant metadata and records
- Reviewing available record sets and their `@id`s
- Extracting records from each record set into DataFrames
- Performing EDA on numeric fields, filtering and normalizing data
- Visualizing distributions and relationships among data fields

This approach enables transparent and reproducible exploration of biomedical tabular datasets. Remember to reference all entities by their `@id` for interoperability, and pay attention to dataset-specific limitations noted in the metadata.

**Note:** With this dataset's small sample size and clinical context, findings are illustrative. For broader analysis or clinical inference, larger and more diverse datasets are needed.